In [1]:
import numpy as np 
import pandas as pd
from pathlib import Path

In [6]:
CLEANED_DATAFRAME_LOCATION = Path('..') / 'datasets' / 'cleaned_dataset.csv'
cleaned_df = pd.read_csv(CLEANED_DATAFRAME_LOCATION)

product_df = (cleaned_df.groupby('Product Name')[['Sales', 'Gross Profit']].sum().sort_values('Sales', ascending=False))
total_products = len(product_df)

# ── 1a. Products contributing 80% of REVENUE ────────────────────────────────
product_df['Revenue Share (%)'] = (product_df['Sales'] / product_df['Sales'].sum() * 100)
product_df['Cumulative Revenue (%)'] = product_df['Revenue Share (%)'].cumsum()

products_for_80_revenue = (product_df[product_df['Cumulative Revenue (%)'] <= 80].shape[0] + 1)
pct_products_80_rev = products_for_80_revenue / total_products * 100

print(f"\nTotal unique products: {total_products}")
print(f"\n--- Products contributing 80% of REVENUE ---")
print(f"{products_for_80_revenue} out of {total_products} products "
      f"({pct_products_80_rev:.1f}%) generate 80% of total revenue\n")

# ── 1b. Products contributing 80% of PROFIT ─────────────────────────────────
product_profit_df = product_df.sort_values('Gross Profit', ascending=False).copy()
product_profit_df['Profit Share (%)'] = (product_profit_df['Gross Profit'] / product_profit_df['Gross Profit'].sum() * 100)
product_profit_df['Cumulative Profit (%)'] = product_profit_df['Profit Share (%)'].cumsum()

products_for_80_profit = (product_profit_df[product_profit_df['Cumulative Profit (%)'] <= 80].shape[0] + 1)
pct_products_80_profit = products_for_80_profit / total_products * 100

print(f"--- Products contributing 80% of PROFIT ---")
print(f" {products_for_80_profit} out of {total_products} products "
      f"({pct_products_80_profit:.1f}%) generate 80% of total profit\n")


Total unique products: 15

--- Products contributing 80% of REVENUE ---
5 out of 15 products (33.3%) generate 80% of total revenue

--- Products contributing 80% of PROFIT ---
 5 out of 15 products (33.3%) generate 80% of total profit



In [12]:
# ═════════════════════════════════════════════════════════════════════════════
# SECTION 2: Congestion-Prone States & Regions
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("SECTION 2: CONGESTION-PRONE STATES & REGIONS")
print("=" * 80)

# ── Region-Level Concentration ───────────────────────────────────────────────
region_df = (
    cleaned_df
    .groupby('Region')
    .agg(
        Total_Orders=('Order ID', 'nunique'),
        Total_Sales=('Sales', 'sum'),
        Total_Profit=('Gross Profit', 'sum'),
        Total_Units=('Units', 'sum'),
        Unique_Customers=('Customer ID', 'nunique'),
        Unique_States=('State/Province', 'nunique')
    )
    .sort_values('Total_Orders', ascending=False)
)

region_df['Order Share (%)'] = region_df['Total_Orders'] / region_df['Total_Orders'].sum() * 100
region_df['Revenue Share (%)'] = region_df['Total_Sales'] / region_df['Total_Sales'].sum() * 100
region_df['Profit Share (%)'] = region_df['Total_Profit'] / region_df['Total_Profit'].sum() * 100
region_df['Orders per State'] = (region_df['Total_Orders'] / region_df['Unique_States']).round(1)

print("\n--- Region-Level Summary ---\n")
print(region_df[['Total_Orders', 'Order Share (%)', 'Revenue Share (%)',
                  'Profit Share (%)', 'Unique_States', 'Orders per State']].round(2).to_string())

# ── State-Level Concentration ────────────────────────────────────────────────
state_df = (
    cleaned_df
    .groupby(['State/Province', 'Region'])
    .agg(
        Total_Orders=('Order ID', 'nunique'),
        Total_Sales=('Sales', 'sum'),
        Total_Profit=('Gross Profit', 'sum'),
        Total_Units=('Units', 'sum')
    )
    .sort_values('Total_Orders', ascending=False)
)

state_df['Order Share (%)'] = state_df['Total_Orders'] / state_df['Total_Orders'].sum() * 100
state_df['Cumulative Order Share (%)'] = state_df['Order Share (%)'].cumsum()
state_df['Revenue Share (%)'] = state_df['Total_Sales'] / state_df['Total_Sales'].sum() * 100

states_for_80_orders = (
    state_df[state_df['Cumulative Order Share (%)'] <= 80].shape[0] + 1
)
total_states = len(state_df)

print(f"\n--- Top Congestion-Prone States ---")
print(f"  {states_for_80_orders} out of {total_states} states "
      f"({states_for_80_orders/total_states*100:.1f}%) handle 80% of all orders\n")


SECTION 2: CONGESTION-PRONE STATES & REGIONS

--- Region-Level Summary ---

          Total_Orders  Order Share (%)  Revenue Share (%)  Profit Share (%)  Unique_States  Orders per State
Region                                                                                                       
Pacific           2744            32.10              32.66             32.63             14             196.0
Atlantic          2476            28.96              29.06             28.87             20             123.8
Interior          1979            23.15              22.60             22.78             14             141.4
Gulf              1350            15.79              15.69             15.73             11             122.7

--- Top Congestion-Prone States ---
  16 out of 59 states (27.1%) handle 80% of all orders



In [13]:
# Congestion risk classification
def classify_congestion_risk(row):
    order_share = row['Order Share (%)']
    if order_share >= 5:
        return 'HIGH'
    elif order_share >= 2:
        return 'MEDIUM'
    else:
        return 'LOW'

state_df['Congestion Risk'] = state_df.apply(classify_congestion_risk, axis=1)
state_df[~state_df['Congestion Risk'].isin(['LOW'])]

,,Total_Orders,Total_Sales,Total_Profit,Total_Units,Order Share (%),Cumulative Order Share (%),Revenue Share (%),Congestion Risk
State/Province,Region,,,,,,,,
California,Pacific,1686,27917.40,18479.42,7667,19.721605,19.721605,19.690143,HIGH
New York,Atlantic,942,15541.03,10222.44,4224,11.018833,30.740437,10.961089,HIGH
Texas,Interior,833,13416.09,8909.53,3724,9.743830,40.484267,9.462369,HIGH
Pennsylvania,Atlantic,487,8027.03,5225.47,2153,5.696573,46.180840,5.661465,HIGH
Illinois,Interior,433,6898.96,4557.68,1845,5.064920,51.245760,4.865837,HIGH
Washington,Pacific,432,6921.15,4566.64,1883,5.053223,56.298982,4.881487,HIGH
Ohio,Atlantic,392,6768.95,4413.03,1759,4.585332,60.884314,4.774141,MEDIUM
Florida,Gulf,319,4804.02,3207.11,1379,3.731431,64.615745,3.388276,MEDIUM
North Carolina,Gulf,216,3450.86,2331.16,983,2.526611,67.142356,2.433892,MEDIUM


In [14]:
# ═════════════════════════════════════════════════════════════════════════════
# SECTION 3: Over-Dependency Risk Analysis
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("SECTION 3: OVER-DEPENDENCY RISK ANALYSIS")
print("=" * 80)

# ── 3a. Product Dependency ───────────────────────────────────────────────────
top_product_name = product_df.index[0]
top_product_rev_share = product_df.iloc[0]['Revenue Share (%)']

product_hhi = (product_df['Revenue Share (%)'] ** 2).sum()
print(f"\n--- 3a. Product Dependency ---")
print(f"  Top product: {top_product_name} ({top_product_rev_share:.1f}%)")
print(f"  Product HHI: {product_hhi:.0f} "
      f"({'HIGH' if product_hhi > 2500 else 'MODERATE' if product_hhi > 1500 else 'LOW'})")

top3_rev_share = product_df['Revenue Share (%)'].head(3).sum()
top3_profit_share = product_profit_df['Profit Share (%)'].head(3).sum()
print(f"  Top 3 products: {top3_rev_share:.1f}% revenue, {top3_profit_share:.1f}% profit")

# ── 3b. Regional Dependency ──────────────────────────────────────────────────
top_region_name = region_df.index[0]
region_hhi = (region_df['Revenue Share (%)'] ** 2).sum()
print(f"\n--- 3b. Regional Dependency ---")
print(f"  Top region: {top_region_name} ({region_df.iloc[0]['Revenue Share (%)']:.1f}%)")
print(f"  Regional HHI: {region_hhi:.0f} "
      f"({'HIGH' if region_hhi > 2500 else 'MODERATE' if region_hhi > 1500 else 'LOW'})")

# ── 3c. Customer Dependency ──────────────────────────────────────────────────
customer_df = (
    cleaned_df
    .groupby('Customer ID')[['Sales', 'Gross Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
)

total_customers = len(customer_df)
customer_df['Revenue Share (%)'] = customer_df['Sales'] / customer_df['Sales'].sum() * 100
customer_df['Cumulative Revenue (%)'] = customer_df['Revenue Share (%)'].cumsum()

customers_for_80_rev = (
    customer_df[customer_df['Cumulative Revenue (%)'] <= 80].shape[0] + 1
)
pct_customers_80_rev = customers_for_80_rev / total_customers * 100
top10_customer_share = customer_df['Revenue Share (%)'].head(10).sum()

print(f"\n--- 3c. Customer Dependency ---")
print(f"  Total customers: {total_customers}")
print(f"  Customers for 80% revenue: {customers_for_80_rev} ({pct_customers_80_rev:.1f}%)")
print(f"  Top 10 customers: {top10_customer_share:.1f}%")


SECTION 3: OVER-DEPENDENCY RISK ANALYSIS

--- 3a. Product Dependency ---
  Top product: Wonka Bar - Triple Dazzle Caramel (20.1%)
  Product HHI: 1766 (MODERATE)
  Top 3 products: 58.7% revenue, 59.3% profit

--- 3b. Regional Dependency ---
  Top region: Pacific (32.7%)
  Regional HHI: 2668 (HIGH)

--- 3c. Customer Dependency ---
  Total customers: 5044
  Customers for 80% revenue: 2532 (50.2%)
  Top 10 customers: 2.0%


In [16]:
print("\n" + "=" * 80)
print("OVERALL PARETO & DEPENDENCY SUMMARY")
print("=" * 80)
print(f"\n  Pareto Distribution:")
print(f"    {pct_products_80_rev:.1f}% of products -> 80% of revenue")
print(f"    {pct_products_80_profit:.1f}% of products -> 80% of profit")
print(f"    {states_for_80_orders/total_states*100:.1f}% of states -> 80% of orders")
print(f"\n  Dependency Risks:")
print(f"    Product HHI:  {product_hhi:.0f} ({'HIGH' if product_hhi > 2500 else 'MODERATE' if product_hhi > 1500 else 'LOW'})")
print(f"    Regional HHI: {region_hhi:.0f} ({'HIGH' if region_hhi > 2500 else 'MODERATE' if region_hhi > 1500 else 'LOW'})")
print(f"    Customer:     {pct_customers_80_rev:.1f}% for 80% rev ({'HIGH' if pct_customers_80_rev < 20 else 'MODERATE' if pct_customers_80_rev < 40 else 'LOW'})")


OVERALL PARETO & DEPENDENCY SUMMARY

  Pareto Distribution:
    33.3% of products -> 80% of revenue
    33.3% of products -> 80% of profit
    27.1% of states -> 80% of orders

  Dependency Risks:
    Product HHI:  1766 (MODERATE)
    Regional HHI: 2668 (HIGH)
    Customer:     50.2% for 80% rev (LOW)
